In [ ]:
!pip install -q -U \
  "transformers==4.55.2" \
  "trl==0.20.0" \
  "peft==0.15.2" \
  "accelerate==1.6.0" \
  "bitsandbytes==0.46.0" \
  datasets


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import torch, transformers, trl
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("trl", trl.__version__)

torch 2.4.1+cu124
transformers 4.55.2
trl 0.20.0


In [ ]:
# V1 학습 코드
import os
os.environ["HF_HOME"] = "/workspace/hf"          # 모델 캐시 → 볼륨
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL    = "Qwen/Qwen3-8B"
DATA_DIR = "/workspace/data"                      # train_all.jsonl, valid_all.jsonl
OUT      = "/workspace/qwen3-8b-workit-v1"

In [ ]:
# 1) 데이터
ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train_all.jsonl",
    "valid": f"{DATA_DIR}/valid_all.jsonl",
})
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 278
    })
    valid: Dataset({
        features: ['messages'],
        num_rows: 64
    })
})


In [ ]:
# 2) 토크나이저 + 4bit 모델
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto",
    trust_remote_code=True, torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# 3) LoRA
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
# 4) 학습 설정 ── 에폭마다 저장 + assistant만 loss
cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=3,                    # 상한 3
    per_device_train_batch_size=2,         # A40 48GB 여유
    gradient_accumulation_steps=4,         # 유효 배치 8  # v -> 1
    learning_rate=2e-4, # v
    lr_-scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    max_length=2048, # v
    packing=False,
    save_strategy="epoch",                 # 에폭마다 체크포인트
    eval_strategy="epoch",                 # 에폭마다 valid loss
    save_total_limit=3,                    # epoch1,2,3 다 보관
    load_best_model_at_end=True,           # 끝나면 valid 최저 로드
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5, # v
    report_to="none",
    assistant_only_loss=True,              # user/system 제외, assistant만 학습
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

tok.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|im_start|>system\n' + message['content'] + '<|im_end|>\n' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% generation %}{{ message['content'] }}{% endgeneration %}"
    "{{ '<|im_end|>\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=ds["train"], eval_dataset=ds["valid"],
    peft_config=lora, processing_class=tok,
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/278 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Epoch,Training Loss,Validation Loss
1,0.653700,0.715162
2,0.313800,0.734162
3,0.151600,0.826617


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


In [ ]:
# 5) best 어댑터 저장
trainer.save_model(f"{OUT}/best")
tok.save_pretrained(f"{OUT}/best")
print("에폭별:", f"{OUT}/checkpoint-*")
print("best  :", f"{OUT}/best")